## Data transformation

This is the second automatically graded exercise for JODA. The objective here is to get our hands dirty with data. 

The context of this particular analysis is a fictional company that routinely runs different machine learning operations. 

We have generated a dataset that has the following columns or properties (to be engineered into features):

* Date
* Department
* ML Task ID
* ML Method
* Task Category
* Model Complexity (Parameters)
* Training Data Size (GB)
* Training Duration (Hours)
* Hardware Used
* Energy Consumption (kWh)
* CO2 Emissions (Kg)
* Cloud Provider

Moreover, there is a secondary dataset that includes information about the energy sources for the different cloud providers:

* Cloud Provider    
* Green Energy


TODO: Install the required packages using requirements.txt or individually

TODO: Import the needed packages   

In [ ]:
# https://github.com/microsoft/vscode-jupyter/wiki/Installing-Python-packages-in-Jupyter-Notebooks

%pip install pandas openpyxl matplotlib
%pip install -r requirements.txt


import pandas as pd
import os

- Read the two data files. **Please note that the data files will be available under the data folder when running the grader.**
- Join the two data frames to add information about the energy sources that the could providers use. [<code>merge()</code>](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) should be useful here.


In [ ]:
print(os.path.isfile('data/cloud-providers.xlsx'))
print(os.path.isfile('data/co2-emissions.xlsx'))

df_co2 = pd.read_excel('data/co2-emissions.xlsx')
df_providers = pd.read_excel('data/cloud-providers.xlsx')
df_merged = df_co2.merge(df_providers, on='Cloud Provider', how='outer')

# df_merged.info()
# df_merged.head()


- Aggregate the data to department level. That is, each row should represents the aggregated values for each department. Notice that you do not need to aggregate each different column, only the ones instructed explicitly.

- Calculate the total of CO2 emissions for each department
- Rename CO2 emission column to <code>co2_emissions_kg</code>   

In [ ]:
df_agg_dept_em = (
    df_merged.groupby('Department').agg(
    {'CO2 Emissions (Kg)': 'sum'}
    )
    .rename(columns={'CO2 Emissions (Kg)': 'co2_emissions_kg'})
    .reset_index()
)

display(df_agg_dept_em)

- Create a function that picks the most common value among in a Pandas Series object
- Pick the most frequent ML method for each department.

In [ ]:

def pick_most_frequent(values):
    # if values.mode().size > 1:
    #     return values.mode().tolist()
    if not values.mode().empty:
        return values.mode()[0]
    else:
        return None

# print(f"Test 1: {pick_most_frequent(pd.Series(['A', 'B', 'B', 'C']))}")
# print(f"Test 2: {pick_most_frequent(pd.Series(['A', 'B', 'B', 'C', 'C']))}")
# print(f"Test 3: {pick_most_frequent(pd.Series([]))}")

df_most_frequent_ml_method = (
    df_merged.groupby('Department').agg(
        {'ML Method': pick_most_frequent}
    )
    .reset_index()
)

df_agg_dept_em = df_agg_dept_em.merge(df_most_frequent_ml_method, on='Department', how='outer')

display(df_agg_dept_em)

- Make sure that the rows are sorted according to CO2 emissions in a way that the department with the largest emissions is first.

In [ ]:
df_agg_dept_em.sort_values('co2_emissions_kg', ascending= False)

- Calculate the CO2 emissions for each department in different Green Energy categories. That is, the resulting dataframe will have as many colums as there are values for Green Energy.

In [ ]:
# df_grouped = (
#     df_merged.groupby(['Department', 'Green Energy'])
#     .agg({'CO2 Emissions (Kg)': 'sum'})
#     .unstack()
# )
# # Flatten the multi-level columns
# df_grouped.columns = df_grouped.columns.droplevel(0)
# # Reset index to make the Department a column
# df_grouped.reset_index(inplace=True)

# print("df_grouped:")
# display(df_grouped)

df_grouped2 = (df_merged
    .pivot_table(
        index='Department', 
        columns='Green Energy', 
        values='CO2 Emissions (Kg)', 
        aggfunc='sum'
    )
    .reset_index()
)

print("df_grouped2:")
display(df_grouped2)

Next, let's try to do something a bit more difficult. That is, calculate department CO2 emissions per energy type. 

One way to achieve this is to use pivot_table() function to create a separate dataframe with the new columns and join (using merge()) that to the main dataframe. We are sure there are even more clever ways. 

- Include the specified columns to the result dataframe, one per each energy type. 

In [ ]:
df_final = (
    df_agg_dept_em.merge(df_grouped2, on='Department', how='outer')
    .sort_values('co2_emissions_kg', ascending=False)
)

display(df_final)

In [ ]:
import os

def ensure_folder_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"Folder '{folder_path}' created.")
    else:
        print(f"Folder '{folder_path}' already exists.")

ensure_folder_exists('results')

TODO Finally, save the results.

In [ ]:
# df_final.to_excel('results/department_co2.xlsx', index=False)
df_final.to_pickle('results/department_co2.pkl')